In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import networkx as nx

# Load the CSV data
data_path = r"C:\Users\olufe\projects\Journal\dataset\HomeC_unsupervise_real_combined_shuffled.csv"
df = pd.read_csv(data_path)

# Preprocessing
X = df.drop(columns=['Label'])  # Features
y = df['Label']  # Labels

# Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Normalize the data (optional but usually recommended for autoencoders)
X_train = (X_train - X_train.mean()) / X_train.std()
X_test = (X_test - X_test.mean()) / X_test.std()

##########################################################################
# ... (previous code remains the same)

# Build the Autoencoder model
input_dim = X_train.shape[1]

def build_autoencoder(input_dim):
    encoder = models.Sequential([
        layers.Dense(64, activation='relu', input_shape=(input_dim,)),
        layers.Dense(32, activation='relu'),
        layers.Dense(16, activation='relu'),
    ])

    decoder = models.Sequential([
        layers.Dense(32, activation='relu', input_shape=(16,)),
        layers.Dense(64, activation='relu'),
        layers.Dense(input_dim, activation='linear'),
    ])

    autoencoder = models.Sequential([encoder, decoder])
    autoencoder.compile(optimizer='adam', loss='mean_squared_error')

    return autoencoder

# Train the Autoencoder
autoencoder = build_autoencoder(input_dim)
autoencoder.fit(X_train, X_train, epochs=50, batch_size=32, validation_data=(X_test, X_test))

# Obtain the encoded representation of the data
encoded_X_train = autoencoder.layers[0].predict(X_train)
encoded_X_test = autoencoder.layers[0].predict(X_test)

# ... (continue with the rest of the code)




#########################################################################

# Build the Autoencoder model
#input_dim = X_train.shape[1]

#def build_autoencoder(input_dim):
#    encoder = models.Sequential([
#        layers.Dense(64, activation='relu', input_shape=(input_dim,)),
#        layers.Dense(32, activation='relu'),
#        layers.Dense(16, activation='relu'),
#    ])

#    decoder = models.Sequential([
#        layers.Dense(32, activation='relu', input_shape=(16,)),
#        layers.Dense(64, activation='relu'),
#        layers.Dense(input_dim, activation='linear'),
#    ])

#    autoencoder = models.Sequential([encoder, decoder])
#    autoencoder.compile(optimizer='adam', loss='mean_squared_error')

#    return autoencoder

# Train the Autoencoder
#autoencoder = build_autoencoder(input_dim)
#autoencoder.fit(X_train, X_train, epochs=50, batch_size=32, validation_data=(X_test, X_test))

# Obtain the encoded representation of the data
#encoded_X_train = autoencoder.layers[0](X_train).numpy()
#encoded_X_test = autoencoder.layers[0](X_test).numpy()
###################################################################################################

# Semi-supervised Ensemble Method
# Combine the encoded data with the labels for semi-supervised training
X_semi_supervised = np.concatenate((encoded_X_train, y_train.values.reshape(-1, 1)), axis=1)

# Graph-Based Method
# Create a similarity graph using the encoded data and k-nearest neighbors
k = 10  # You can adjust this hyperparameter
graph = nx.Graph()
for i, point in enumerate(encoded_X_train):
    distances = np.linalg.norm(encoded_X_train - point, axis=1)
    nearest_neighbors = np.argsort(distances)[1:k + 1]  # Exclude the point itself
    for neighbor_idx in nearest_neighbors:
        graph.add_edge(i, neighbor_idx)

# Evaluate the model using performance metrics
def evaluate_model(y_true, y_pred):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_pred)
    return accuracy, precision, recall, f1, auc

# Fit and evaluate Semi-Supervised Ensemble Method
semi_supervised_classifier = RandomForestClassifier()  # You can use any other classifier here
semi_supervised_classifier.fit(X_semi_supervised[:, :-1], X_semi_supervised[:, -1])
y_pred_semi_supervised = semi_supervised_classifier.predict(encoded_X_test)
accuracy, precision, recall, f1, auc = evaluate_model(y_test, y_pred_semi_supervised)
print("Semi-Supervised Ensemble Method:")
print(f"Accuracy: {accuracy}, Precision: {precision}, Recall: {recall}, F1-Score: {f1}, AUC: {auc}")

# Fit and evaluate Graph-Based Method
y_pred_graph_based = np.zeros(len(y_test))
for i, point in enumerate(encoded_X_test):
    neighbors = list(graph.neighbors(i))
    neighbor_labels = [y_train.iloc[neighbor] for neighbor in neighbors]
    y_pred_graph_based[i] = int(sum(neighbor_labels) >= len(neighbors) / 2)

accuracy, precision, recall, f1, auc = evaluate_model(y_test, y_pred_graph_based)
print("Graph-Based Method:")
print(f"Accuracy: {accuracy}, Precision: {precision}, Recall: {recall}, F1-Score: {f1}, AUC: {auc}")


Epoch 1/50
1314/1314 [==============================] - 6s 4ms/step - loss: 0.6469 - val_loss: 0.5404
Epoch 2/50
1314/1314 [==============================] - 4s 3ms/step - loss: 0.5125 - val_loss: 0.4890
Epoch 3/50
1314/1314 [==============================] - 4s 3ms/step - loss: 0.4788 - val_loss: 0.4692
Epoch 4/50
1314/1314 [==============================] - 4s 3ms/step - loss: 0.4619 - val_loss: 0.4537
Epoch 5/50
1314/1314 [==============================] - 5s 4ms/step - loss: 0.4507 - val_loss: 0.4479
Epoch 6/50
1314/1314 [==============================] - 4s 3ms/step - loss: 0.4446 - val_loss: 0.4408
Epoch 7/50
1314/1314 [==============================] - 4s 3ms/step - loss: 0.4397 - val_loss: 0.4372
Epoch 8/50
1314/1314 [==============================] - 5s 4ms/step - loss: 0.4357 - val_loss: 0.4328
Epoch 9/50
1314/1314 [==============================] - 5s 4ms/step - loss: 0.4329 - val_loss: 0.4294
Epoch 10/50
1314/1314 [==============================] - 4s 3ms/step - loss: 0.430